# InstaSearch Training Workflow

This notebook demonstrates how to load the proteomics dataset, prepare the dual-encoder model, and run a training epoch using the repository code in `src/`. It also shows how to execute the full `train_script.py` command-line workflow.

In [ ]:
import os
import sys

# Setup path
repo_root = os.path.abspath(os.path.join(os.getcwd()))
sys.path.insert(0, repo_root)

print('Repo root:', repo_root)
print('Python executable:', sys.executable)

In [ ]:
# Import core libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from typing import List, Tuple, Optional
from torch.utils.data import Dataset, random_split

# --- Configuration Constants ---
MAX_PEAKS = 500
MAX_PEPTIDE_LEN = 42
NUM_AA = 26
RADIANT_BASE = 10000.0
SEED = 42
D_MODEL = 512
N_HEADS = 4
D_FF = 1024
N_LAYERS = 4
EMBED_DIM = 128
DROPOUT = 0.1
INIT_TEMP = 0.1

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print('Torch version:', torch.__version__)
print('Device:', DEVICE)

In [ ]:
# --- Transformer Components (joint_model.py) ---
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + self.drop1(attn_out)
        x = x + self.drop2(self.ffn(self.norm2(x)))
        return x

class ProjectionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout), nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, p=2, dim=-1)

print('Loaded transformer components')

In [ ]:
# --- Spectrum Encoder Components (spectrum_encoder.py) ---
class MultiScalePeakEmbedding(nn.Module):
    """Multi-scale sinusoidal embedding based on Voronov et. al."""

    def __init__(self, h_size: int, dropout: float = 0, float_dtype: torch.dtype | str = torch.float64) -> None:
        super().__init__()
        self.h_size = h_size
        self.float_dtype = getattr(torch, float_dtype, None) if isinstance(float_dtype, str) else float_dtype
        if self.float_dtype is None:
            raise ValueError(f"Unknown torch dtype string: {float_dtype}")

        self.mlp = nn.Sequential(
            nn.Linear(h_size, h_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h_size, h_size),
            nn.Dropout(dropout),
        )

        self.head = nn.Sequential(
            nn.Linear(h_size + 1, h_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h_size, h_size),
            nn.Dropout(dropout),
        )

        freqs = 2 * np.pi / torch.logspace(-2, -3, int(h_size / 2), dtype=self.float_dtype)
        self.register_buffer("freqs", freqs)

    def forward(self, spectra: torch.Tensor) -> torch.Tensor:
        """Encode peaks."""
        mz_values, intensities = spectra[:, :, [0]], spectra[:, :, [1]]
        x = self.encode_mass(mz_values)
        x = self.mlp(x)
        x = torch.cat([x, intensities], axis=2)
        return self.head(x)

    def encode_mass(self, x: torch.Tensor) -> torch.Tensor:
        """Encode mz."""
        x = self.freqs[None, None, :] * x
        x = torch.cat([torch.sin(x), torch.cos(x)], axis=2)
        return x.float()


class SpectrumEncoder(nn.Module):
    """Encodes an (mz, intensity) peak matrix together with precursor information into a fixed-size embedding."""
    MAX_CHARGE: int = 4

    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, n_layers=N_LAYERS, embed_dim=EMBED_DIM, dropout=DROPOUT):
        super().__init__()
        self.peak_encoder = MultiScalePeakEmbedding(d_model, dropout=dropout)
        self.charge_embedding = nn.Embedding(
            num_embeddings=self.MAX_CHARGE + 1,
            embedding_dim=d_model
        )
        self.precursor_proj = nn.Linear(2 * d_model, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.layers = nn.ModuleList([TransformerEncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.proj_head = ProjectionHead(d_model, d_model * 2, embed_dim)

    def _encode_precursor(self, precursors: torch.Tensor) -> torch.Tensor:
        """precursors: [B, 2] where [:, 0] = precursor_mz, [:, 1] = charge"""
        mz_peak = precursors[:, :1].unsqueeze(-1)
        ones = torch.ones_like(mz_peak)
        mz_peak = torch.cat([mz_peak, ones], dim=-1)
        mz_emb = self.peak_encoder(mz_peak).squeeze(1)

        charge_raw = precursors[:, 1]
        charge_raw = torch.nan_to_num(charge_raw, nan=0.0, posinf=0.0, neginf=0.0)
        charge = charge_raw.long().clamp(0, self.MAX_CHARGE)
        charge_emb = self.charge_embedding(charge)

        combined = torch.cat([mz_emb, charge_emb], dim=-1)
        return self.precursor_proj(combined)

    def forward(self, x, precursors):
        """x: [B, MAX_PEAKS, 2], precursors: [B, 2]  Returns: [B, EMBED_DIM]"""
        B = x.size(0)
        is_pad = (x.abs().sum(dim=-1) == 0)
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        pad_mask = torch.cat([cls_mask, is_pad], dim=1)

        x = self.peak_encoder(x)
        pre_emb = self._encode_precursor(precursors).unsqueeze(1)
        cls = self.cls_token.expand(B, -1, -1) + pre_emb
        x = torch.cat([cls, x], dim=1)

        for layer in self.layers:
            x = layer(x, key_padding_mask=pad_mask)

        return self.proj_head(self.final_norm(x[:, 0, :]))

print('Loaded spectrum encoder')

In [ ]:
# --- Peptide Encoder Components (peptide_encoder.py) ---
AA_VOCAB = {aa: idx for idx, aa in enumerate([
    "<PAD>", "A", "C", "D", "E", "F", "G", "H", "I", "K",
    "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W",
    "Y", "B", "Z", "X", "U", "O"
])}

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=43):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class PeptideEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, n_layers=N_LAYERS, embed_dim=EMBED_DIM, max_len=MAX_PEPTIDE_LEN, dropout=DROPOUT):
        super().__init__()
        self.aa_embed = nn.Embedding(NUM_AA, d_model, padding_idx=0)
        self.aa_pos_embed = PositionalEncoding(d_model, dropout=dropout, max_len=max_len + 1)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.proj_head = ProjectionHead(d_model, d_model * 2, embed_dim)

    def forward(self, tokens):
        """tokens: [B, MAX_PEPTIDE_LEN] (int64, 0 = pad) Returns: [B, EMBED_DIM]"""
        B = tokens.size(0)
        is_pad = (tokens == 0)
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=tokens.device)
        pad_mask = torch.cat([cls_mask, is_pad], dim=1)

        x = self.aa_embed(tokens)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.aa_pos_embed(x)

        for layer in self.layers:
            x = layer(x, key_padding_mask=pad_mask)

        return self.proj_head(self.final_norm(x[:, 0, :]))

print('Loaded peptide encoder')

In [ ]:
# --- Dataset Class (data/dataset.py) ---
class SpectraPeptideDataset(Dataset):
    def __init__(self, specs, peps, pres):
        self.specs = torch.tensor(specs, dtype=torch.float32)
        self.peps = torch.tensor(peps, dtype=torch.int64)
        self.pres = torch.tensor(pres, dtype=torch.float32)
    
    def __len__(self):
        return len(self.specs)
    
    def __getitem__(self, i):
        return self.specs[i], self.peps[i], self.pres[i]

print('Loaded dataset class')

In [ ]:
# --- Preprocessing Functions (data/preprocess.py) ---
def preprocess_spectrum(mz_array, intensity_array, max_peaks=MAX_PEAKS):
    if mz_array is None or intensity_array is None:
        return np.zeros((max_peaks, 2))

    mz_arr = np.array(mz_array, dtype=np.float64)
    int_arr = np.array(intensity_array, dtype=np.float64)

    if len(mz_arr) == 0 or len(int_arr) == 0:
        return np.zeros((max_peaks, 2))

    if len(mz_arr) != len(int_arr):
        min_len = min(len(mz_arr), len(int_arr))
        mz_arr = mz_arr[:min_len]
        int_arr = int_arr[:min_len]

    if len(mz_arr) > max_peaks:
        top_idx = np.argsort(int_arr)[-max_peaks:]
        top_idx = np.sort(top_idx)
        mz_arr, int_arr = mz_arr[top_idx], int_arr[top_idx]

    l2_norm = np.sqrt(np.sum(int_arr ** 2))
    int_arr_norm = int_arr / l2_norm if l2_norm > 0 else int_arr

    spectrum = np.zeros((max_peaks, 2))
    spectrum[:len(mz_arr), 0] = mz_arr
    spectrum[:len(int_arr), 1] = int_arr_norm

    return spectrum


def preprocess_peptide(sequence, max_len=MAX_PEPTIDE_LEN):
    if not isinstance(sequence, str) or len(sequence) == 0:
        return np.zeros(max_len, dtype=np.int64)

    sequence = sequence[:max_len]
    indices = [AA_VOCAB.get(aa, 0) for aa in sequence]
    indices += [0] * (max_len - len(indices))
    return np.array(indices, dtype=np.int64)


def preprocess_dataset(df):
    required_cols = ['mz_array', 'intensity_array', 'sequence']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    n = len(df)
    spec_tensors = np.zeros((n, MAX_PEAKS, 2))
    pep_tokens = np.zeros((n, MAX_PEPTIDE_LEN), dtype=np.int64)
    precursors = np.zeros((n, 2))

    valid_indices = []
    for i, (_, row) in enumerate(df.iterrows()):
        try:
            seq = row['sequence']
            mz = row['mz_array']
            intensity = row['intensity_array']

            if not isinstance(seq, str) or len(seq) == 0:
                continue
            if not isinstance(mz, (list, np.ndarray)) and pd.isna(mz):
                continue
            if not isinstance(intensity, (list, np.ndarray)) and pd.isna(intensity):
                continue

            spec_tensors[i] = preprocess_spectrum(mz, intensity)
            pep_tokens[i] = preprocess_peptide(seq)
            charge_val = row.get('precursor_charge', 0)
            if charge_val is None or (isinstance(charge_val, float) and np.isnan(charge_val)):
                charge_val = 0
            precursors[i] = [row.get('precursor_mz', 0), charge_val]
            valid_indices.append(i)
        except Exception as e:
            print(f"Error processing row {i}: {e}")
            continue

    return spec_tensors[valid_indices], pep_tokens[valid_indices], precursors[valid_indices]

print('Loaded preprocessing functions')

In [ ]:
# --- Loss Function (training/loss.py) ---
class CLIPContrastiveLoss(nn.Module):
    def __init__(self, init_temp=INIT_TEMP, label_smoothing=0.1):
        super().__init__()
        self.log_temp = nn.Parameter(torch.tensor(math.log(init_temp)))
        self.label_smoothing = label_smoothing
        
    def forward(self, z_spec, z_pep):
        temp = self.log_temp.clamp(min=math.log(0.04), max=math.log(0.5)).exp()
        logits = (z_spec @ z_pep.T) / temp
        labels = torch.arange(logits.size(0), device=logits.device)
        loss = (F.cross_entropy(logits, labels, label_smoothing=self.label_smoothing) + 
                F.cross_entropy(logits.T, labels, label_smoothing=self.label_smoothing)) / 2
        acc = ((logits.argmax(dim=1) == labels).float().mean() + 
               (logits.argmax(dim=0) == labels).float().mean()) / 2
        return loss, acc

print('Loaded loss function')

In [ ]:
# --- Training Functions (training/train.py) ---
def train_epoch(model_spec, model_pep, loader, loss_fn, opt, scaler):
    model_spec.train(); model_pep.train()
    total_loss, total_acc = 0.0, 0.0

    for specs, peps, pres in loader:
        specs, peps, pres = specs.to(DEVICE), peps.to(DEVICE), pres.to(DEVICE)
        opt.zero_grad()

        if DEVICE.type == "cuda" and scaler is not None:
            with torch.amp.autocast("cuda"):
                z_spec = model_spec(specs, pres)
                z_pep = model_pep(peps)
                loss, acc = loss_fn(z_spec, z_pep)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm(
                list(model_spec.parameters()) + list(model_pep.parameters()),
                max_norm=1.0
            )
            scaler.step(opt)
            scaler.update()
        else:
            z_spec = model_spec(specs, pres)
            z_pep = model_pep(peps)
            loss, acc = loss_fn(z_spec, z_pep)
            loss.backward()
            opt.step()

        total_loss += loss.item(); total_acc += acc.item()
    return total_loss/len(loader), total_acc/len(loader)


@torch.no_grad()
def validate(model_spec, model_pep, loader, loss_fn):
    model_spec.eval(); model_pep.eval()
    total_loss, total_acc = 0.0, 0.0

    if len(loader) == 0:
        return 0.0, 0.0

    for specs, peps, pres in loader:
        specs, peps, pres = specs.to(DEVICE), peps.to(DEVICE), pres.to(DEVICE)

        if DEVICE.type == "cuda":
            with torch.amp.autocast("cuda"):
                z_spec = model_spec(specs, pres)
                z_pep = model_pep(peps)
                loss, acc = loss_fn(z_spec, z_pep)
        else:
            z_spec = model_spec(specs, pres)
            z_pep = model_pep(peps)
            loss, acc = loss_fn(z_spec, z_pep)

        total_loss += loss.item(); total_acc += acc.item()
    return total_loss/len(loader), total_acc/len(loader)

print('Loaded training functions')

In [ ]:
# Load and preprocess dataset
dataset_name = 'InstaDeepAI/ms_ninespecies_benchmark'
split = 'train[:2000]'

print('Loading dataset:', dataset_name, split)
raw_ds = load_dataset(dataset_name, split=split)
df = raw_ds.to_pandas()

specs, peps, pres = preprocess_dataset(df)
print('Preprocessed examples:', len(specs))

dataset = SpectraPeptideDataset(specs, peps, pres)
print('Dataset length:', len(dataset))

In [ ]:
batch_size = 64
train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

model_spec = SpectrumEncoder(d_model=256, n_heads=4, d_ff=512, n_layers=2, embed_dim=64).to(DEVICE)
model_pep = PeptideEncoder(d_model=256, n_heads=4, d_ff=512, n_layers=2, embed_dim=64).to(DEVICE)
loss_fn = CLIPContrastiveLoss(init_temp=0.1).to(DEVICE)
optimizer = torch.optim.AdamW(list(model_spec.parameters()) + list(model_pep.parameters()) + [loss_fn.log_temp], lr=1e-4, weight_decay=5e-3)

print('Model spec params:', sum(p.numel() for p in model_spec.parameters()))
print('Model peptide params:', sum(p.numel() for p in model_pep.parameters()))

In [ ]:
from torch.utils.data import random_split

# Split the dataset the same way the training script does
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False)

scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
num_epochs = 3

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model_spec, model_pep, train_loader, loss_fn, optimizer, scaler)
    val_loss, val_acc = validate(model_spec, model_pep, val_loader, loss_fn)

    history['loss'].append(train_loss)
    history['acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch {epoch + 1}/{num_epochs} | '
        f'Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}'
    )

## Training Visualization

Define and use plotting functions directly for training metrics, similarity matrices, and embedding visualization.


In [ ]:
# Plotting functions

def plot_metrics(history):
    fig, ax1 = plt.subplots(figsize=(10, 5))

    color = 'tab:red'
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss', color=color)
    ax1.plot(history['loss'], color=color, marker='o', label='Train Loss')
    if 'val_loss' in history:
        ax1.plot(history['val_loss'], color=color, marker='x', linestyle='--', label='Val Loss')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.legend(loc='upper left')

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Accuracy', color=color)
    ax2.plot(history['acc'], color=color, marker='s', label='Train Accuracy')
    if 'val_acc' in history:
        ax2.plot(history['val_acc'], color=color, marker='d', linestyle='--', label='Val Accuracy')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.legend(loc='upper right')

    plt.title("Training & Validation Metrics")
    fig.tight_layout()
    plt.show()


def plot_similarity_matrix(z_spec, z_pep):
    # Compute cosine similarity matrix
    sim = (z_spec @ z_pep.T).cpu().numpy()

    plt.figure(figsize=(8, 6))
    sns.heatmap(sim, annot=False, cmap='viridis')
    plt.title("Spectrum-Peptide Similarity Matrix (Batch)")
    plt.xlabel("Peptide Index")
    plt.ylabel("Spectrum Index")
    plt.show()


def plot_embeddings(z_spec, z_pep, title="Embedding Visualization (t-SNE)"):
    # Concatenate embeddings
    z_s = z_spec.cpu().numpy()
    z_p = z_pep.cpu().numpy()
    z_all = np.concatenate([z_s, z_p], axis=0)

    # Run t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    z_2d = tsne.fit_transform(z_all)

    # Split back
    z_s_2d = z_2d[:len(z_s)]
    z_p_2d = z_2d[len(z_s):]

    plt.figure(figsize=(10, 8))
    plt.scatter(z_s_2d[:, 0], z_s_2d[:, 1], alpha=0.6, label='Spectra', c='tab:red', marker='o')
    plt.scatter(z_p_2d[:, 0], z_p_2d[:, 1], alpha=0.6, label='Peptides', c='tab:blue', marker='x')

    # Draw lines between matching pairs
    for i in range(len(z_s_2d)):
        plt.plot([z_s_2d[i, 0], z_p_2d[i, 0]], [z_s_2d[i, 1], z_p_2d[i, 1]], 'k-', alpha=0.1)

    plt.legend()
    plt.title(title)
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.grid(True, alpha=0.3)
    plt.show()


# Plot training and validation metrics
plot_metrics(history)

# Visualize a single batch similarity matrix and embeddings
batch = next(iter(val_loader))
specs_batch, peps_batch, pres_batch = batch
z_spec = model_spec(specs_batch.to(DEVICE), pres_batch.to(DEVICE))
z_pep = model_pep(peps_batch.to(DEVICE))

plot_similarity_matrix(z_spec, z_pep)
plot_embeddings(z_spec, z_pep)


## Run the full training script

The repository also includes `train_script.py`, which implements the full CLI-based training workflow with dataset loading, checkpoint saving, and validation. Run this command from the repository root to train with a larger subset:

```bash
python train_script.py --dataset InstaDeepAI/ms_ninespecies_benchmark --dataset_split train[:20000] --batch_size 128 --num_epochs 10 --output_dir ./checkpoints
```